In [1]:
import pandas as pd 

df = pd.read_csv('country_info.csv')

/Users/justinwyrley/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Add duck data

In [44]:
df = pd.read_csv('country_info_updated.csv')
df

,alphabetic_country_rank,name,url,capital,languages,largest_religion,area_total,population,gdp_nominal_total,gdp_nominal_per_capita,currency,time_zone,observes_dst,calling_code,flag_path,anthem_path,duck_pop_rank,continent
0,1,Afghanistan,https://en.wikipedia.org/wiki/Afghanistan,Kabul,Pashto Dari,Dharmic Buddhism Hinduism Jainism Sikhism Isla...,"652,864 km²",35 million,NaN,NaN,افغانى (AFN),UTC +4:30,0,+93,countries/Afghanistan_flag.png,NaN,NaN,Asia
1,2,Albania,https://en.wikipedia.org/wiki/Albania,Tirana,Albanian,50.67% Islam 45.86% Sunni 4.81% Bektashism 16....,"28,748 km²",2.4 million,NaN,NaN,(ALL),UTC +1,1,+355,countries/Albania_flag.png,countries/Albania_anthem.ogg,NaN,Europe
2,3,Algeria,https://en.wikipedia.org/wiki/Algeria,Algiers,Arabic Berber,99% Sunni Islam ( official ) <1% others,"2,381,741 km²",47.4 million,NaN,NaN,(DZD),UTC +1,0,+213,countries/Algeria_flag.png,countries/Algeria_anthem.mp3,54.0,Africa
3,4,Andorra,https://en.wikipedia.org/wiki/Andorra,Andorra la Vella,Catalan 44.1%,90.8% Christianity 85.5% Catholicism ( officia...,467.63 km²,"89,000",NaN,NaN,€ (EUR),UTC +1,1,+376,countries/Andorra_flag.png,countries/Andorra_anthem.mp3,NaN,Europe
4,5,Angola,https://en.wikipedia.org/wiki/Angola,Luanda,"Chokwe , Kimbundu , Kikongo , Oshiwambo , Luch...",79.1% Christianity 44.2% Catholic 34.9% Protes...,"1,246,700 km²",36.6 million,NaN,NaN,(AOA),UTC +1,0,+244,countries/Angola_flag.png,countries/Angola_anthem.ogg,NaN,Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,192,Venezuela,https://en.wikipedia.org/wiki/Venezuela,Caracas,Spanish,92.6% Christianity 80.5% Catholicism 11.2% Pro...,"916,445 km²",31.8 million,NaN,NaN,(VED) (official),UTC -4,0,+58,countries/Venezuela_flag.png,countries/Venezuela_anthem.ogg,NaN,South America
192,193,Vietnam,https://en.wikipedia.org/wiki/Vietnam,Hanoi,Vietnamese,86.32% no religion / folk 6.1% Catholicism 4.7...,"331,344.82 km²",102.3 million,NaN,NaN,₫ (VND),UTC +7,0,+84,countries/Vietnam_flag.png,countries/Vietnam_anthem.ogg,2.0,Asia
193,194,Yemen,https://en.wikipedia.org/wiki/Yemen,Sanaa,Arabic,~99.1% Islam ( official ) 65% Sunni 35% Zaydi ...,"455,503 km²",32.7 million,NaN,NaN,(YER),UTC +3,0,+967,countries/Yemen_flag.png,countries/Yemen_anthem.ogg,NaN,Asia
194,195,Zambia,https://en.wikipedia.org/wiki/Zambia,Lusaka,English,95.5% Christianity ( official ) 2.7% Islam 1.8...,"752,617 km²",20.2 million,NaN,NaN,(ZMW),UTC +2,0,+260,countries/Zambia_flag.png,countries/Zambia_anthem.mp3,NaN,Africa


In [2]:
duck = pd.read_csv('random_data/duck_data.csv', dtype={'Population Count': str, 'Ducks Per Capita (per 1.000)': str})
duck['name'] = duck['Country/Territory']

# use index as duck population ranking
duck.reset_index(inplace=True)
for i in range(len(duck)):
    duck.at[i, 'index'] = duck.at[i, 'index'] + 1

duck['duck_pop_rank'] = duck['index'].astype('Int64')

# Only keep population ranking maybe can find a use for other data
duck = duck[['duck_pop_rank', 'name']]

df = pd.merge(df, duck, on='name', how='left')

## Manually add missing into 

In [3]:
df.at[0, "calling_code"] = "+93" # Afghanistan
df.at[97, "time_zone"] = "UTC +1" # Luxembourg
df.at[20, "languages"] = "bosnian" # Bosnia and Herzegovina
df.at[108, "area_total"] = "33,843 km 2" # Moldova
df.at[29, "largest_religion"] = "Christianity" # Canada
df.at[36, "largest_religion"] = "Islam" # Comoros
df.at[61, "largest_religion"] = "Christianity" # Germany
df.at[70, "largest_religion"] = "Christianity" # Honduras
df.at[81, "largest_religion"] = "Shinto" # Japan
df.at[116, "largest_religion"] = "Christianity" # Nauru
df.at[136, "largest_religion"] = "Christianity" # Portugal
df.at[141, "largest_religion"] = "Christianity" # Russia
df.at[179, "largest_religion"] = "Islam" # Turkey
df.at[138, "time_zone"] = "UTC ±0" # Republic of Ireland


## Number rounding

In [4]:
import re

def format_population(pop_str):
    if pd.isna(pop_str): return ""
    
    # Extract numeric part
    multiplier = 1_000_000 if 'million' in pop_str.lower() else 1
    nums = re.findall(r'\d+(?:,\d+)*(?:\.\d+)?', pop_str)
    if not nums: return ""
    
    val = float(nums[0].replace(',', '')) * multiplier
    
    # Rounding logic
    if val >= 1_000_000:
        rounded_m = round(val / 1_000_000, 1)
        return f"{int(rounded_m) if rounded_m == int(rounded_m) else rounded_m} million"
    elif val >= 100_000:
        return f"{int(round(val, -5)):,}"
    elif val >= 1_000:
        return f"{int(round(val, -3)):,}"
    else:
        return str(int(val))

df['population'] = df['population'].apply(format_population)

## Alphabetical country ranking

In [5]:
# Use index to rank countries alphabetically
df.reset_index(inplace=True)
df.rename(columns={'index': 'alphabetic_country_rank'}, inplace=True)
for i in range(len(df)):
    df.at[i, 'alphabetic_country_rank'] = df.at[i, 'alphabetic_country_rank'] + 1

## Clean up timezones

In [6]:
def format_timezone(text):
    if pd.isna(text):
        return text
    
    # 1. Standardise dashes and minus signs
    text = text.replace('–', '-').replace('—', '-').replace('−', '-')
    
    # 2. Slice off anything inside or after parentheses
    text = text.split('(')[0]
    
    # 3. Clean and format the numerical offsets
    # Group 1: Optional sign (+, -, ±)
    # Group 2: Hours (\d+)
    # Group 3: Optional fraction/minutes (e.g., :30, .5, :00)
    def clean_offset(match):
        sign = match.group(1).strip() if match.group(1) else ''
        hours = int(match.group(2)) # Converts '01' to 1, '00' to 0
        fraction = match.group(3) if match.group(3) else ''
        
        # If minutes are exactly :00 or .0, drop them completely
        if fraction in [':00', '.0', '']:
            return f"{sign}{hours}"
        return f"{sign}{hours}{fraction}"
        
    text = re.sub(r'([±+-]?)\s*(\d+)((?:[:.]\d+)?)', clean_offset, text)
    
    # 4. Remove all unwanted alphabetical text
    # We keep 'UTC', 'to', and 'and' to preserve ranges and multiple zones
    words_to_keep = {'utc', 'to', 'and'}
    def filter_words(match):
        word = match.group(0)
        if word.lower() in words_to_keep:
            return 'UTC' if word.lower() == 'utc' else word.lower()
        return ''
        
    text = re.sub(r'[A-Za-z]+', filter_words, text)
    
    # 5. Tidy up spacing and ensure 'UTC' always has a trailing space
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'UTC(?=[^\s])', 'UTC ', text) 
    
    # 6. Remove any dangling punctuation at the very end
    text = text.strip(' ,;/')
    
    return text

# Apply the transformation
df['time_zone'] = df['time_zone'].apply(format_timezone)

zero_replacements = {
    'UTC +0': 'UTC ±0',
    'UTC -0': 'UTC ±0',
    'UTC 0': 'UTC ±0'
}

# Iterate through the dictionary and replace substrings directly
for old_val, new_val in zero_replacements.items():
    df['time_zone'] = df['time_zone'].str.replace(old_val, new_val, regex=False)

## Clean up area total 

In [7]:
def format_area(area_str):
    if pd.isna(area_str): return ""
    
    # 1. Target the number explicitly associated with 'km'
    # This safely ignores 'sq mi' regardless of whether it is inside or outside brackets
    match = re.search(r'([\d,]+(?:\.\d+)?)\s*km', str(area_str), re.IGNORECASE)
    
    if match:
        # Extract and clean the matched number
        num_str = match.group(1).replace(',', '')
        val = float(num_str)
        
        # Format neatly: integers get commas, floats keep their decimals
        if val.is_integer():
            return f"{int(val):,} km²"
        else:
            return f"{val:,} km²"
        
    return ""

# Apply the formatting
df['area_total'] = df['area_total'].apply(format_area)

# Clean currency

In [8]:
def format_currency(text):
    if pd.isna(text): 
        return ""
    
    # 1. Find all contents inside any brackets
    matches = re.findall(r'\(([^)]+)\)', str(text))
    
    if not matches:
        return ""
        
    formatted_items = []
    for match in matches:
        # Clean up any messy whitespace inside the brackets
        inner_text = match.strip()
        
        # 2. Check if the text consists strictly of standard letters
        # This perfectly isolates codes like 'EUR', 'USD', 'AFN'
        if re.fullmatch(r'[A-Za-z]+', inner_text):
            formatted_items.append(f"({inner_text})")
        else:
            # Anything with symbols, numbers, or non-Latin scripts (e.g. €, R$, £, ₼)
            # drops the brackets and stands alone
            formatted_items.append(inner_text)
            
    # Join the processed items together with a space
    return ' '.join(formatted_items)

# Apply the formatting to the dataset
df['currency'] = df['currency'].apply(format_currency)

# Clean capital 

In [9]:
df['capital'] = df['capital'].str.extract(r'^(\D+)')[0].str.strip()

## Add continents 

In [30]:
continents = pd.read_csv('random_data/list-of-countries-by-continent-2026.csv')
continents.rename(columns={'country': 'name'}, inplace=True)
continents = continents[['name', 'continent']]
df = pd.merge(df, continents[['name', 'continent']], on='name', how='left')

In [ ]:
save = df.to_csv('country_info_updated.csv', index=False)